In [1]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_milo
# python -m ipykernel install --user --name scrna_cartography_milo --display-name "milo"

In [2]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# milo
import pertpy as pt
milo = pt.tl.Milo()
import mudata as md
from mudata import MuData

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np

# plotting
import matplotlib.pyplot as plt 

import random
from sklearn.metrics.pairwise import euclidean_distances
from sklearn_ann.kneighbors.annoy import AnnoyTransformer 
from collections import defaultdict
# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import diff_genes as dg
import misc as mi
import dg_milo as dm

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Create directories
mi.create_directories(dir_path = str(here('data/differential_abundance/files')))
mi.create_directories(dir_path = str(here('data/differential_abundance/de_analysis_groups')))
mi.create_directories(dir_path = str(here('data/differential_abundance/de_analysis_overall')))
mi.create_directories(dir_path = str(here('data/differential_abundance/plots')))
mi.create_directories(dir_path = str(here('data/differential_abundance/objects')))

/work/islet_cartography_scrna/data/differential_abundance/files Directory already exists!
/work/islet_cartography_scrna/data/differential_abundance/de_analysis_groups Directory already exists!
/work/islet_cartography_scrna/data/differential_abundance/de_analysis_overall Directory already exists!
/work/islet_cartography_scrna/data/differential_abundance/plots Directory already exists!
/work/islet_cartography_scrna/data/differential_abundance/objects Directory already exists!


In [4]:
# Paths
base_dir = str(here('data/differential_abundance/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
de_dir = os.path.join(base_dir, 'de_analysis_groups') 
de_overall_dir = os.path.join(base_dir, 'de_analysis_overall') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

In [5]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

In [6]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))
mdata = milo.load(adata)
sc.pp.neighbors(mdata["rna"], use_rep="X_latent_1", transformer=AnnoyTransformer(666))
milo.make_nhoods(mdata, prop=0.1)
mdata = milo.count_nhoods(mdata, sample_col="ic_id_platform_adjusted_sample")
mdata["rna"].obs["sample_cell_count"] = (
    mdata["rna"].obs.groupby("ic_id_platform_adjusted_sample")["ic_id_platform_adjusted_sample"]
    .transform("count")
)
milo.annotate_nhoods(mdata, anno_col="manual_annotation")
mdata["milo"].var["nhood_annotation_origianl"] = mdata["milo"].var["nhood_annotation"]
mdata["milo"].var["nhood_annotation"] = mdata["milo"].var["nhood_annotation"].cat.add_categories("mixed")
mdata["milo"].var.loc[mdata["milo"].var["nhood_annotation_frac"] < 0.5, "nhood_annotation"] = "mixed"
mdata.write(os.path.join(objects_dir,"mdata_666_annotation.h5mu"))

In [7]:
ref_cells = mdata["rna"].obs_names[mdata["rna"].obs["nhood_ixs_refined"] == 1] # Get reference cells, in neighbourhood order
nhood_to_ref = dict(enumerate(ref_cells)) # dictionary with nhood number 
nhoods = mdata["rna"].obsm["nhoods"] # Get the neighborhoods
cell_names = mdata["rna"].obs_names # Get the cell names
rows, cols = nhoods.nonzero() # numeric neighbourhood id (column index)
df = pd.DataFrame({
    "cell": cell_names[rows],
    "nhood_id": cols,                    # numeric
    "neighborhood": ref_cells[cols],     # barcode
})
df = df.merge(adata.obs[['disease_harmonized', 'ic_id_platform_adjusted_sample', 'manual_annotation']].reset_index().rename(columns={"index": "cell"}), how = "left", on = "cell")
df = df.merge(mdata['milo'].var.rename(columns={"index_cell": "neighborhood"}), how = "left", on = "neighborhood")

In [8]:
# 1) samples per neighborhood
samples_per_nhood = df.groupby("neighborhood")["ic_id_platform_adjusted_sample"].nunique()
assert samples_per_nhood.median() > 5, "Too few samples per neighborhood"

# 2) condition mixing
conds_per_nhood = df.groupby("neighborhood")["disease_harmonized"].nunique()
assert (conds_per_nhood >= 2).mean() > 0.9, "Too many neighborhoods lack condition mixing"

# 3) neighborhood size
nhood_sizes = df.groupby("neighborhood").size()
assert nhood_sizes.min() > 50, "Neighborhoods too small"
assert nhood_sizes.max() < 5000, "Neighborhoods too large"

# 4) sample dominance
df_counts = (
    df.groupby(["neighborhood", "ic_id_platform_adjusted_sample"])
    .size()
    .reset_index(name="n")
)

df_counts["frac"] = df_counts.groupby("neighborhood")["n"].transform(lambda x: x / x.sum())
max_frac = df_counts.groupby("neighborhood")["frac"].max()

assert max_frac.median() < 0.8, "Neighborhoods dominated by single sample"

conds_per_nhood = df.groupby("neighborhood")["disease_harmonized"].nunique()
assert (conds_per_nhood >= 2).mean() > 0.9, "Too many neighborhoods lack condition mixing"

# Annotation specificity 
assert (mdata["milo"].var["nhood_annotation_frac"] > 0.8).mean() > 0.9, "Too many neighborhoods are specific to one cell type"

print("All sanity checks passed.")

All sanity checks passed.


In [9]:
df.to_csv(os.path.join(files_dir, "cells_in_neighborhood.csv"))

#### Differential expression

In [10]:
res_adata = dm.milo_da_nhoods(
    mdata,
    design="~sample_cell_count+disease_harmonized",
    model_contrasts="disease_harmonizedt2d-disease_harmonizednd"
)

res_adata.var.to_csv(os.path.join(files_dir, "t2d_vs_nd.csv"))

Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.46 seconds.

Fitting dispersions...
... done in 4.03 seconds.

Fitting dispersion trend curve...
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/pydeseq2/dds.py:820: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.53 seconds.

Fitting MAP dispersions...
... done in 3.94 seconds.

Fitting LFCs...
... done in 5.91 seconds.

Calculating cook's distance...
... done in 0.57 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 2.70 seconds.



Log2 fold change & Wald test p-value: disease_harmonized t2d vs nd
       baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
0      1.250610       -0.464238  0.417551 -1.111812  0.266219       NaN
1      1.900522       -0.161648  0.316886 -0.510112  0.609973  0.819765
2      1.386757       -0.621573  0.371353 -1.673805  0.094169       NaN
3      1.380261        0.394345  0.330246  1.194097  0.232440       NaN
4      1.794386       -0.209640  0.318664 -0.657871  0.510621  0.757185
...         ...             ...       ...       ...       ...       ...
32092  1.922618       -1.565166  0.443135 -3.532029  0.000412  0.015205
32093  1.739062       -0.749478  0.469671 -1.595750  0.110545  0.340098
32094  2.455023       -1.387585  0.556104 -2.495187  0.012589  0.094670
32095  2.600829       -0.905083  0.382284 -2.367564  0.017906  0.116358
32096  1.604131       -0.676070  0.355438 -1.902079  0.057161  0.229724

[32097 rows x 6 columns]


In [11]:
res_adata = dm.milo_da_nhoods(
    mdata,
    design="~sample_cell_count+disease_harmonized",
    model_contrasts="disease_harmonizedpre-disease_harmonizednd"
)

res_adata.var.to_csv(os.path.join(files_dir, "pre_vs_nd.csv"))

Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.47 seconds.

Fitting dispersions...
... done in 3.82 seconds.

Fitting dispersion trend curve...
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/pydeseq2/dds.py:820: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.54 seconds.

Fitting MAP dispersions...
... done in 3.79 seconds.

Fitting LFCs...
... done in 5.81 seconds.

Calculating cook's distance...
... done in 0.58 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 2.89 seconds.



Log2 fold change & Wald test p-value: disease_harmonized pre vs nd
       baseMean  log2FoldChange     lfcSE      stat        pvalue      padj
0      1.250610        0.796062  0.652815  1.219429  2.226815e-01  0.468775
1      1.900522        0.743968  0.474766  1.567020  1.171100e-01  0.336142
2      1.386757        0.792665  0.553199  1.432875  1.518935e-01  0.382907
3      1.380261        0.697631  0.542933  1.284928  1.988173e-01  0.442725
4      1.794386       -0.356582  0.521019 -0.684394  4.937264e-01  0.702787
...         ...             ...       ...       ...           ...       ...
32092  1.922618       -4.159522  0.804466 -5.170536  2.334237e-07  0.000595
32093  1.739062       -1.650400  0.814611 -2.025997  4.276512e-02  0.197529
32094  2.455023       -3.606429  0.999563 -3.608006  3.085592e-04  0.013402
32095  2.600829       -2.278745  0.661381 -3.445436  5.701392e-04  0.019042
32096  1.604131       -1.230287  0.592817 -2.075325  3.795642e-02  0.184981

[32097 rows x 6 colu

In [ ]:
res_adata.var